# NLOS UWB Distance Comparison

NLOS 환경에서 카메라 GT 기준 이론 거리와 UWB 실제 측정 거리(`Dist`)를 비교합니다.

카메라 높이: `1.60 m`, UWB 앵커 높이: `1.88 m`이므로 높이 차이는 `0.28 m`입니다.

이론 거리는 다음 3D 거리식으로 계산합니다.

`theoretical_distance = sqrt((gt_x - anchor_x)^2 + (gt_y - anchor_y)^2 + (camera_z - anchor_z)^2)`

## 넣어야 하는 파일

아래 두 파일을 `D:/uwb-config/data` 폴더에 넣고, 다음 설정 셀에서 파일명을 맞춰주세요.

카메라 GT 파일: `camera_gt_nlos.csv`

```csv
#Packet,x,y
1,0.50,1.20
2,0.55,1.23
3,0.60,1.26
```

UWB 실제 거리 파일: `uwb_nlos.csv`

```csv
#Packet,ANC,Dist
1,1,1.48
2,1,1.52
3,1,1.55
```

`x`, `y`, `anchor_x`, `anchor_y`는 반드시 같은 좌표계 기준이어야 합니다. 카메라 GT가 cm 단위이면 `GT_UNIT = "cm"`로 바꾸세요.

In [ ]:
from pathlib import Path
import math

import numpy as np
import pandas as pd


repo_root = Path.cwd()
if repo_root.name.lower() == "calibration":
    repo_root = repo_root.parent

data_dir = repo_root / "data"

# 1) Put these files in D:/uwb-config/data.
CAMERA_GT_FILE = data_dir / "camera_gt_nlos.csv"
UWB_FILE = data_dir / "uwb_nlos.csv"

# 2) CSV column names.
MERGE_KEY = "#Packet"
GT_X_COL = "x"
GT_Y_COL = "y"
ANCHOR_COL = "ANC"
UWB_DIST_COL = "Dist"

# 3) Unit and height settings.
GT_UNIT = "m"  # Use "cm" if camera GT x/y are centimeters.
CAMERA_HEIGHT_M = 1.60
ANCHOR_HEIGHT_M = 1.88

# 4) Anchor x/y coordinates in the same coordinate system as camera GT.
# Replace x and y with your measured anchor location.
DEFAULT_ANCHOR_ID = 1
ANCHORS = {
    1: {"x": 0.0, "y": 0.0, "z": ANCHOR_HEIGHT_M},
    # 2: {"x": 2.0, "y": 0.0, "z": ANCHOR_HEIGHT_M},
    # 3: {"x": 0.0, "y": 2.0, "z": ANCHOR_HEIGHT_M},
}

# 5) If your UWB CSV already contains only NLOS rows, keep this as None.
# Example for a mixed LOS/NLOS CSV: NLOS_FILTER_COLUMN = "Condition", NLOS_FILTER_VALUE = "NLOS"
NLOS_FILTER_COLUMN = None
NLOS_FILTER_VALUE = "NLOS"

height_diff_m = abs(ANCHOR_HEIGHT_M - CAMERA_HEIGHT_M)
height_diff_m

## 계산 함수

아래 셀은 CSV를 읽고, 카메라 GT와 UWB 로그를 `#Packet` 기준으로 맞춘 뒤 이론 거리와 실제 거리의 오차를 계산합니다.

In [ ]:
def unit_scale(unit):
    scales = {"m": 1.0, "cm": 0.01, "mm": 0.001}
    if unit not in scales:
        raise ValueError(f"Unsupported unit: {unit}. Use one of {list(scales)}.")
    return scales[unit]


def require_columns(df, columns, name):
    missing = [column for column in columns if column not in df.columns]
    if missing:
        raise ValueError(f"{name} is missing columns: {missing}")


def normalize_merge_key(series):
    return series.astype(str).str.strip().str.replace(r"\.0$", "", regex=True)


def load_camera_gt(path):
    if not path.exists():
        raise FileNotFoundError(f"Camera GT file not found: {path}")

    gt = pd.read_csv(path)
    require_columns(gt, [MERGE_KEY, GT_X_COL, GT_Y_COL], "Camera GT CSV")

    scale = unit_scale(GT_UNIT)
    gt[MERGE_KEY] = normalize_merge_key(gt[MERGE_KEY])
    gt["gt_x_m"] = pd.to_numeric(gt[GT_X_COL], errors="coerce") * scale
    gt["gt_y_m"] = pd.to_numeric(gt[GT_Y_COL], errors="coerce") * scale
    gt = gt.dropna(subset=["gt_x_m", "gt_y_m"])

    return gt[[MERGE_KEY, "gt_x_m", "gt_y_m"]]


def load_uwb(path):
    if not path.exists():
        raise FileNotFoundError(f"UWB file not found: {path}")

    uwb = pd.read_csv(path)
    require_columns(uwb, [MERGE_KEY, UWB_DIST_COL], "UWB CSV")

    if ANCHOR_COL not in uwb.columns:
        uwb[ANCHOR_COL] = DEFAULT_ANCHOR_ID

    if NLOS_FILTER_COLUMN and NLOS_FILTER_COLUMN in uwb.columns:
        is_nlos = uwb[NLOS_FILTER_COLUMN].astype(str).str.upper() == str(NLOS_FILTER_VALUE).upper()
        uwb = uwb[is_nlos].copy()

    uwb[MERGE_KEY] = normalize_merge_key(uwb[MERGE_KEY])
    uwb[UWB_DIST_COL] = pd.to_numeric(uwb[UWB_DIST_COL], errors="coerce")
    uwb = uwb.dropna(subset=[UWB_DIST_COL])

    return uwb[[MERGE_KEY, ANCHOR_COL, UWB_DIST_COL]]


def get_anchor(anchor_id):
    try:
        key = int(anchor_id)
    except (TypeError, ValueError):
        key = DEFAULT_ANCHOR_ID

    if key not in ANCHORS:
        raise KeyError(f"Anchor {key} is not defined in ANCHORS.")

    return ANCHORS[key]


def theoretical_distance(row):
    anchor = get_anchor(row[ANCHOR_COL])
    dx = row["gt_x_m"] - float(anchor["x"])
    dy = row["gt_y_m"] - float(anchor["y"])
    dz = CAMERA_HEIGHT_M - float(anchor.get("z", ANCHOR_HEIGHT_M))

    return math.sqrt(dx * dx + dy * dy + dz * dz)


def build_comparison(camera_gt_file, uwb_file):
    gt = load_camera_gt(camera_gt_file)
    uwb = load_uwb(uwb_file)
    merged = pd.merge(uwb, gt, on=MERGE_KEY, how="inner")

    if merged.empty:
        raise ValueError("No matched rows. Check that #Packet values match in both CSV files.")

    merged["theoretical_distance_m"] = merged.apply(theoretical_distance, axis=1)
    merged["uwb_distance_m"] = merged[UWB_DIST_COL]
    merged["error_m"] = merged["uwb_distance_m"] - merged["theoretical_distance_m"]
    merged["abs_error_m"] = merged["error_m"].abs()
    merged["percent_error"] = np.where(
        merged["theoretical_distance_m"] > 0,
        merged["error_m"] / merged["theoretical_distance_m"] * 100.0,
        np.nan,
    )

    return merged[
        [
            MERGE_KEY,
            ANCHOR_COL,
            "gt_x_m",
            "gt_y_m",
            "theoretical_distance_m",
            "uwb_distance_m",
            "error_m",
            "abs_error_m",
            "percent_error",
        ]
    ]

## 비교 실행

`ANCHORS`의 `x`, `y`를 실제 앵커 위치로 바꾼 뒤 실행하세요.

In [ ]:
comparison = build_comparison(CAMERA_GT_FILE, UWB_FILE)
comparison.head(10)

In [ ]:
summary = pd.Series(
    {
        "camera_height_m": CAMERA_HEIGHT_M,
        "anchor_height_m": ANCHOR_HEIGHT_M,
        "height_difference_m": height_diff_m,
        "sample_count": len(comparison),
        "mean_theoretical_distance_m": comparison["theoretical_distance_m"].mean(),
        "mean_uwb_distance_m": comparison["uwb_distance_m"].mean(),
        "mean_error_m": comparison["error_m"].mean(),
        "mae_m": comparison["abs_error_m"].mean(),
        "rmse_m": np.sqrt(np.mean(np.square(comparison["error_m"]))),
        "max_abs_error_m": comparison["abs_error_m"].max(),
    },
    name="value",
).to_frame()

summary

## 결과 저장

비교 결과 CSV는 `D:/uwb-config/data/nlos_distance_comparison_result.csv`로 저장됩니다.

In [ ]:
OUTPUT_FILE = data_dir / "nlos_distance_comparison_result.csv"
comparison.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
OUTPUT_FILE

## 그래프

`matplotlib`이 설치된 환경이면 이론 거리와 실제 거리, 오차 변화를 그래프로 확인할 수 있습니다.

In [ ]:
try:
    import matplotlib.pyplot as plt

    x_axis = comparison[MERGE_KEY]

    plt.figure(figsize=(12, 4))
    plt.plot(x_axis, comparison["theoretical_distance_m"], label="Theoretical distance")
    plt.plot(x_axis, comparison["uwb_distance_m"], label="UWB distance")
    plt.xlabel(MERGE_KEY)
    plt.ylabel("Distance (m)")
    plt.title("NLOS Theoretical vs UWB Distance")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(12, 3))
    plt.plot(x_axis, comparison["error_m"], label="UWB - theoretical")
    plt.axhline(0, color="black", linewidth=1)
    plt.xlabel(MERGE_KEY)
    plt.ylabel("Error (m)")
    plt.title("NLOS Distance Error")
    plt.legend()
    plt.grid(True)
    plt.show()
except ModuleNotFoundError:
    print("matplotlib is not installed. The comparison table and saved CSV are still available.")